# Bias Simulation Scenarios
**Master's Thesis | Data Science**

## Five experiments, all using the trained LSTM (weights never change):

### Imputation experiments
- **Exp 1** Black patients: fill missing slots with Black patient median (within-group imputation)
- **Exp 2** White patients: remove measurements to match real Black observation rate

### Bias simulation scenarios
- **Scenario A** SpO2 systematic downward shift for Black patients (1–5 points)
- **Scenario B** Complete temperature removal for Black patients
- **Scenario C** Heart rate Gaussian noise injection for Black patients

All experiments are test-time interventions only. Model weights are never changed.


---
## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
from pathlib import Path
from sklearn.metrics import average_precision_score, precision_recall_curve
from sklearn.model_selection import StratifiedKFold
import torch
import torch.nn as nn

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

OUT_DIR  = Path('/home/ino/thesis_outputs')
SIM_DIR  = OUT_DIR / 'simulation'
SIM_DIR.mkdir(exist_ok=True)

VITAL_NAMES  = ['heart_rate', 'resp_rate', 'spo2', 'temperature', 'map']
VITAL_IDX    = {v: i for i, v in enumerate(VITAL_NAMES)}
KEEP_RACES   = ['White', 'Black', 'Hispanic', 'Asian']
N_FOLDS      = 5
RANDOM_STATE = 42
N_BOOTSTRAP  = 2000

# Vital index shortcuts
V_HR   = VITAL_IDX['heart_rate']   # 0
V_SPO2 = VITAL_IDX['spo2']         # 2
V_TEMP = VITAL_IDX['temperature']  # 3

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Output: {SIM_DIR}')

---
## 2. Load & align data

In [ ]:
# Exact same alignment as thesis_imputation_auprc.py
cv_stay_ids = np.load(OUT_DIR / 'cv_stay_ids.npy')
print(f'CV stay_ids: {len(cv_stay_ids):,}')

cohort_full = pd.read_csv(OUT_DIR / 'cohort.csv')
cohort_full = cohort_full.set_index('stay_id')
cohort_cv   = cohort_full.loc[cv_stay_ids].reset_index()

cohort_full_reset = pd.read_csv(OUT_DIR / 'cohort.csv').reset_index(drop=True)
stay_id_to_ts_idx = {sid: i for i, sid in
                     enumerate(cohort_full_reset['stay_id'].values)}

X_ts_full = np.load(OUT_DIR / 'X_ts.npy')
M_ts_full = np.load(OUT_DIR / 'M_ts.npy')

ts_indices = np.array([stay_id_to_ts_idx[sid] for sid in cv_stay_ids])
X_ts_cv    = X_ts_full[ts_indices]
M_ts_cv    = M_ts_full[ts_indices]

keep_mask = cohort_cv['race_group'].isin(KEEP_RACES).values
cohort    = cohort_cv[keep_mask].reset_index(drop=True)
kept_idx  = np.where(keep_mask)[0]

X_ts    = X_ts_cv[kept_idx]   # (N, 24, 5)
M_ts    = M_ts_cv[kept_idx]   # (N, 24, 5)

y    = cohort['hospital_expire_flag'].values.astype(np.int32)
demo = cohort[['race_group', 'gender', 'age_group']].reset_index(drop=True)
race_arr = demo['race_group'].values

mask_black = (race_arr == 'Black')
mask_white = (race_arr == 'White')

print(f'Cohort: {len(cohort):,}  |  Mortality: {y.mean():.1%}')
print(cohort['race_group'].value_counts().to_string())

---
## 3. LSTM infrastructure

In [ ]:
class MortalityLSTM(nn.Module):
    def __init__(self, input_size=10, hidden_size=128, num_layers=3, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size, hidden_size=hidden_size,
            num_layers=num_layers, batch_first=True,
            dropout=dropout, bidirectional=True
        )
        self.layer_norm = nn.LayerNorm(hidden_size * 2)
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size*2, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64),            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        last   = self.layer_norm(out[:, -1, :])
        return self.classifier(self.dropout(last)).squeeze(1)


def lstm_predict(model, X_input, batch_size=1024):
    model.eval()
    probs = []
    with torch.no_grad():
        for s in range(0, len(X_input), batch_size):
            xb = torch.tensor(X_input[s:s+batch_size],
                              dtype=torch.float32).to(device)
            probs.append(torch.sigmoid(model(xb)).cpu().numpy())
    return np.concatenate(probs)


def normalise_ts(X_input, X_train):
    """Normalise using training fold statistics."""
    ts_mean = X_train.mean(axis=(0,1), keepdims=True)
    ts_std  = X_train.std(axis=(0,1),  keepdims=True) + 1e-8
    return (X_input - ts_mean) / ts_std


def build_input(X_norm, M):
    return np.concatenate([X_norm, M], axis=2).astype(np.float32)


def load_lstm(fold):
    model = MortalityLSTM()
    state = torch.load(OUT_DIR / f'lstm_fold{fold+1}.pt', map_location=device)
    model.load_state_dict(state)
    return model.to(device)


# Build fold splits
strat_label = (demo['race_group'].astype(str) + '_' +
               cohort['hospital_expire_flag'].astype(str)).values
skf         = StratifiedKFold(n_splits=N_FOLDS, shuffle=True,
                              random_state=RANDOM_STATE)
fold_splits  = list(skf.split(np.zeros(len(cohort)), strat_label))
print(f'Fold sizes: {[len(t) for _,t in fold_splits]}')


def run_experiment(modifier_fn, label='experiment'):
    oof = np.zeros(len(cohort))
    for fold, (train_idx, test_idx) in enumerate(fold_splits):
        model    = load_lstm(fold)
        X_te     = X_ts[test_idx].copy()
        M_te     = M_ts[test_idx].copy()
        race_te  = race_arr[test_idx]
        X_mod, M_mod = modifier_fn(X_te, M_te, race_te, train_idx)
        X_norm   = normalise_ts(X_mod, X_ts[train_idx])
        oof[test_idx] = lstm_predict(model, build_input(X_norm, M_mod))
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print(f'  {label} — fold {fold+1} done')
    return oof


def auprc(y_true, probs):
    return average_precision_score(y_true, probs)


def bootstrap_ci(y_true, probs, n_boot=N_BOOTSTRAP, seed=42):
    rng = np.random.default_rng(seed)
    n   = len(y_true)
    scores = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        ys, ps = y_true[idx], probs[idx]
        if ys.sum() == 0 or ys.sum() == n: continue
        scores.append(average_precision_score(ys, ps))
    s = np.array(scores)
    return s.mean(), np.percentile(s, 2.5), np.percentile(s, 97.5)


def bootstrap_delta_p(y_true, p_base, p_exp, n_boot=N_BOOTSTRAP, seed=0):
    rng = np.random.default_rng(seed)
    n   = len(y_true)
    deltas = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        ys  = y_true[idx]
        if ys.sum() == 0 or ys.sum() == n: continue
        deltas.append(auprc(ys, p_exp[idx]) - auprc(ys, p_base[idx]))
    d = np.array(deltas)
    return 2 * min(np.mean(d <= 0), np.mean(d >= 0))


print('All helpers defined.')

---
## 4. Baseline (clean LSTM)

In [ ]:
def no_change(X_te, M_te, race_te, train_idx):
    return X_te, M_te

print('Running baseline...')
oof_baseline = run_experiment(no_change, label='baseline')

bl_black = auprc(y[mask_black], oof_baseline[mask_black])
bl_white = auprc(y[mask_white], oof_baseline[mask_white])
bl_all   = auprc(y, oof_baseline)

print(f'\nBaseline AUPRC:')
print(f'  Overall  {bl_all:.4f}')
print(f'  Black    {bl_black:.4f}')
print(f'  White    {bl_white:.4f}')
print(f'  Gap (White - Black)  {bl_white - bl_black:+.4f}')

---
## 5. Imputation Experiment 1: Black patients: fill missing with BLACK median


In [ ]:
# Pre-compute per-fold Black training medians
black_median = {}  # black_median[fold][v][h] = median value or None
black_rate   = {}  # black_rate[fold][v][h]   = observation rate

for fold, (train_idx, _) in enumerate(fold_splits):
    black_tr  = train_idx[race_arr[train_idx] == 'Black']
    X_bl      = X_ts[black_tr]
    M_bl      = M_ts[black_tr]
    fold_med  = {}
    fold_rate = {}
    for v in range(len(VITAL_NAMES)):
        fold_med[v]  = {}
        fold_rate[v] = {}
        for h in range(24):
            obs = M_bl[:, h, v] == 1
            fold_med[v][h]  = float(np.median(X_bl[:, h, v][obs])) if obs.any() else None
            fold_rate[v][h] = float(M_bl[:, h, v].mean())
    black_median[fold] = fold_med
    black_rate[fold]   = fold_rate

print('Black training medians computed.')

In [ ]:
def impute_black_with_black_median(X_te, M_te, race_te, train_idx):
    fold = [f for f, (tr, _) in enumerate(fold_splits) if np.array_equal(tr, train_idx)][0]
    bmed = black_median[fold]
    X_out = X_te.copy()
    M_out = M_te.copy()
    black_loc = np.where(race_te == 'Black')[0]
    for li in black_loc:
        for v in range(len(VITAL_NAMES)):
            for h in range(24):
                if M_out[li, h, v] == 0 and bmed[v][h] is not None:
                    X_out[li, h, v] = bmed[v][h]
                    M_out[li, h, v] = 1.0
    return X_out, M_out


print('Running Exp 1: Black missing -> Black median imputation...')
oof_exp1 = run_experiment(impute_black_with_black_median, label='Exp1')

e1_black = auprc(y[mask_black], oof_exp1[mask_black])
e1_white = auprc(y[mask_white], oof_exp1[mask_white])
p1 = bootstrap_delta_p(y[mask_black], oof_baseline[mask_black], oof_exp1[mask_black])

print(f'\nExp 1 — Black patients imputed with Black median:')
print(f'  Black  baseline={bl_black:.4f}  after={e1_black:.4f}  '
      f'delta={e1_black-bl_black:+.4f}  p={p1:.4f}')
print(f'  White  unchanged={e1_white:.4f} (sanity check, should match baseline)')

---
## 6. Imputation Experiment 2: White patients: remove measurements to match real Black rate


In [ ]:
rng_exp2 = np.random.default_rng(42)

def degrade_white_to_black_rate(X_te, M_te, race_te, train_idx):
    """
    For White test patients: randomly remove measurements so their
    per-vital per-hour observation rate matches the Black training rate.
    Removed slots get population median value and M=0.
    """
    fold  = [f for f, (tr, _) in enumerate(fold_splits) if np.array_equal(tr, train_idx)][0]
    brate = black_rate[fold]
    X_out = X_te.copy()
    M_out = M_te.copy()
    white_loc = np.where(race_te == 'White')[0]
    for li in white_loc:
        for v in range(len(VITAL_NAMES)):
            for h in range(24):
                if M_out[li, h, v] == 1:
                    # Keep with probability = black_rate; remove otherwise
                    if rng_exp2.random() > brate[v][h]:
                        M_out[li, h, v] = 0.0
                        X_out[li, h, v] = float(
                            np.median(X_ts[train_idx][:, h, v]))
    return X_out, M_out


print('Running Exp 2: White measurements degraded to Black rate...')
oof_exp2 = run_experiment(degrade_white_to_black_rate, label='Exp2')

e2_black = auprc(y[mask_black], oof_exp2[mask_black])
e2_white = auprc(y[mask_white], oof_exp2[mask_white])
p2 = bootstrap_delta_p(y[mask_white], oof_baseline[mask_white], oof_exp2[mask_white])

print(f'\nExp 2 — White patients degraded to Black measurement rate:')
print(f'  White  baseline={bl_white:.4f}  after={e2_white:.4f}  '
      f'delta={e2_white-bl_white:+.4f}  p={p2:.4f}')
print(f'  Black  unchanged={e2_black:.4f} (sanity check)')

---
## 7. Scenario A: SpO2 Systematic Downward Shift for Black Patients

We test shifts of 1, 2, 3, 4, 5 percentage points downward.

In [ ]:
SPO2_SHIFTS = [0, 1, 2, 3, 4, 5]   # 0 = baseline (no shift)

# SpO2 bounds from original notebook
SPO2_MIN, SPO2_MAX = 50.0, 100.0

def make_spo2_shift_fn(shift):
    def fn(X_te, M_te, race_te, train_idx):
        X_out = X_te.copy()
        black_loc = np.where(race_te == 'Black')[0]
        for li in black_loc:
            for h in range(24):
                if M_te[li, h, V_SPO2] == 1:  # only shift observed values
                    new_val = X_out[li, h, V_SPO2] - shift
                    X_out[li, h, V_SPO2] = np.clip(new_val, SPO2_MIN, SPO2_MAX)
        return X_out, M_te
    return fn


print('Running Scenario A: SpO2 shift (0–5 points)...')
spo2_results = {}   # shift -> {'black': auprc, 'white': auprc}

for shift in SPO2_SHIFTS:
    print(f'\n  Shift = {shift} points')
    oof = run_experiment(make_spo2_shift_fn(shift), label=f'SpO2-{shift}')
    spo2_results[shift] = {
        'black': auprc(y[mask_black], oof[mask_black]),
        'white': auprc(y[mask_white], oof[mask_white]),
        'oof':   oof,
    }
    print(f'    Black AUPRC={spo2_results[shift]["black"]:.4f}  '
          f'White AUPRC={spo2_results[shift]["white"]:.4f}')

# Bootstrap p-values vs baseline (shift=0)
print('\nBootstrap p-values (vs baseline):')
for shift in SPO2_SHIFTS[1:]:
    p = bootstrap_delta_p(
        y[mask_black],
        spo2_results[0]['oof'][mask_black],
        spo2_results[shift]['oof'][mask_black]
    )
    spo2_results[shift]['p'] = p
    print(f'  Shift={shift}  Black AUPRC={spo2_results[shift]["black"]:.4f}  '
          f'delta={spo2_results[shift]["black"]-bl_black:+.4f}  p={p:.4f}')
spo2_results[0]['p'] = 1.0

---
## 8. Scenario B: Complete Temperature Removal for Black Patients

In [ ]:
def remove_temperature_black(X_te, M_te, race_te, train_idx):
    """
    Set all temperature measurements to missing for Black patients.
    Replace value with training population median for that hour.
    """
    X_out = X_te.copy()
    M_out = M_te.copy()
    black_loc = np.where(race_te == 'Black')[0]
    for li in black_loc:
        for h in range(24):
            if M_out[li, h, V_TEMP] == 1:   # only touch observed slots
                M_out[li, h, V_TEMP] = 0.0
                X_out[li, h, V_TEMP] = float(
                    np.median(X_ts[train_idx][:, h, V_TEMP]))
    return X_out, M_out


print('Running Scenario B: complete temperature removal for Black patients...')
oof_scen_b = run_experiment(remove_temperature_black, label='ScenB')

sb_black = auprc(y[mask_black], oof_scen_b[mask_black])
sb_white = auprc(y[mask_white], oof_scen_b[mask_white])
pb = bootstrap_delta_p(y[mask_black], oof_baseline[mask_black], oof_scen_b[mask_black])

print(f'\nScenario B — Temperature removed for Black patients:')
print(f'  Black  baseline={bl_black:.4f}  after={sb_black:.4f}  '
      f'delta={sb_black-bl_black:+.4f}  p={pb:.4f}')
print(f'  White  unchanged={sb_white:.4f}')

---
## 9. Scenario C: Heart Rate Gaussian Noise Injection for Black Patients

We test noise levels of 0, 5, 10, 15, 20 bpm standard deviation.

In [ ]:
HR_NOISE_LEVELS = [0, 5, 10, 15, 20]  # bpm standard deviation
HR_MIN, HR_MAX  = 20.0, 300.0

rng_hr = np.random.default_rng(99)

def make_hr_noise_fn(std_bpm):
    def fn(X_te, M_te, race_te, train_idx):
        X_out = X_te.copy()
        black_loc = np.where(race_te == 'Black')[0]
        for li in black_loc:
            for h in range(24):
                if M_te[li, h, V_HR] == 1:   # only add noise to observed
                    noise = rng_hr.normal(0, std_bpm)
                    X_out[li, h, V_HR] = np.clip(
                        X_out[li, h, V_HR] + noise, HR_MIN, HR_MAX)
        return X_out, M_te
    return fn


print('Running Scenario C: heart rate noise injection (0–20 bpm)...')
hr_results = {}

for std in HR_NOISE_LEVELS:
    print(f'\n  Noise std = {std} bpm')
    oof = run_experiment(make_hr_noise_fn(std), label=f'HR-{std}')
    hr_results[std] = {
        'black': auprc(y[mask_black], oof[mask_black]),
        'white': auprc(y[mask_white], oof[mask_white]),
        'oof':   oof,
    }
    print(f'    Black AUPRC={hr_results[std]["black"]:.4f}  '
          f'White AUPRC={hr_results[std]["white"]:.4f}')

print('\nBootstrap p-values vs baseline (std=0):')
for std in HR_NOISE_LEVELS[1:]:
    p = bootstrap_delta_p(
        y[mask_black],
        hr_results[0]['oof'][mask_black],
        hr_results[std]['oof'][mask_black]
    )
    hr_results[std]['p'] = p
    print(f'  Std={std:2d} bpm  Black AUPRC={hr_results[std]["black"]:.4f}  '
          f'delta={hr_results[std]["black"]-bl_black:+.4f}  p={p:.4f}')
hr_results[0]['p'] = 1.0

---
## 10. Results & Figures

In [ ]:
# Figure 1: Imputation experiments summary 
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Exp 1 Black imputed with Black median
ax = axes[0]
vals   = [bl_black, e1_black]
labels = ['Baseline\n(original)', 'Exp 1\n(Black median imputed)']
colors = ['#D65F5F', '#6ACC65']
bars   = ax.bar(labels, vals, color=colors, edgecolor='white', width=0.5, alpha=0.85)
for bar, val in zip(bars, vals):
    ax.text(bar.get_x()+bar.get_width()/2, val+0.003,
            f'{val:.4f}', ha='center', fontsize=11, fontweight='bold')
ax.set_ylabel('AUPRC')
ax.set_title('Exp 1 — Black patients\nMissing slots filled with Black median')
ax.text(0.5, 0.05, f'delta={e1_black-bl_black:+.4f}  p={p1:.3f}',
        ha='center', transform=ax.transAxes, fontsize=10,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))
ax.set_ylim(0, max(vals)*1.15)

# Exp 2 White degraded to Black rate
ax = axes[1]
vals   = [bl_white, e2_white]
labels = ['Baseline\n(original)', 'Exp 2\n(Black rate applied)']
colors = ['#4878CF', '#D65F5F']
bars   = ax.bar(labels, vals, color=colors, edgecolor='white', width=0.5, alpha=0.85)
for bar, val in zip(bars, vals):
    ax.text(bar.get_x()+bar.get_width()/2, val+0.003,
            f'{val:.4f}', ha='center', fontsize=11, fontweight='bold')
ax.set_ylabel('AUPRC')
ax.set_title('Exp 2 — White patients\nMeasurements removed to match Black rate')
ax.text(0.5, 0.05, f'delta={e2_white-bl_white:+.4f}  p={p2:.3f}',
        ha='center', transform=ax.transAxes, fontsize=10,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))
ax.set_ylim(0, max(vals)*1.15)

plt.suptitle('Imputation experiments v2 — LSTM (weights unchanged)', fontsize=13)
plt.tight_layout()
plt.savefig(SIM_DIR / 'fig_imputation_v2.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_imputation_v2.png')

In [ ]:
# Figure 2: SpO2 shift sensitivity curve 
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

shifts       = SPO2_SHIFTS
black_auprc  = [spo2_results[s]['black'] for s in shifts]
white_auprc  = [spo2_results[s]['white'] for s in shifts]
p_vals       = [spo2_results[s].get('p', 1.0) for s in shifts]

ax = axes[0]
ax.plot(shifts, black_auprc, 'o-', color='#D65F5F', linewidth=2.5,
        markersize=8, label='Black patients (shifted)')
ax.axhline(bl_white, color='#4878CF', linestyle='--', linewidth=1.5,
           label=f'White baseline ({bl_white:.3f})')
ax.axhline(bl_black, color='#D65F5F', linestyle=':', linewidth=1.5,
           label=f'Black baseline ({bl_black:.3f})')
# Mark significant points
for s, ap, p in zip(shifts, black_auprc, p_vals):
    if p < 0.05:
        ax.plot(s, ap, '*', color='black', markersize=14, zorder=5)
ax.set_xlabel('SpO2 shift (percentage points subtracted from Black patients)')
ax.set_ylabel('AUPRC')
ax.set_title('Scenario A — SpO2 downward shift\n(* = significant vs baseline, p<0.05)')
ax.legend(fontsize=9)
ax.set_xticks(shifts)

ax = axes[1]
gaps = [bl_white - spo2_results[s]['black'] for s in shifts]
ax.bar(shifts, gaps, color='#c0392b', alpha=0.75, edgecolor='white')
ax.axhline(bl_white - bl_black, color='gray', linestyle='--',
           linewidth=1.5, label=f'Baseline gap ({bl_white-bl_black:.3f})')
ax.set_xlabel('SpO2 shift (points)')
ax.set_ylabel('White AUPRC − Black AUPRC')
ax.set_title('Growing disparity gap as SpO2 bias increases')
ax.legend(fontsize=9)
ax.set_xticks(shifts)

plt.suptitle('Scenario A: SpO2 systematic bias against Black patients', fontsize=13)
plt.tight_layout()
plt.savefig(SIM_DIR / 'fig_scenario_a_spo2.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_scenario_a_spo2.png')

In [ ]:
# Figure 3: Scenario B Temperature removal
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Bar chart
ax = axes[0]
vals   = [bl_black, sb_black]
labels = ['Baseline\n(original)', 'Scenario B\n(temperature removed)']
colors = ['#D65F5F', '#8e44ad']
bars   = ax.bar(labels, vals, color=colors, edgecolor='white', width=0.5, alpha=0.85)
for bar, val in zip(bars, vals):
    ax.text(bar.get_x()+bar.get_width()/2, val+0.003,
            f'{val:.4f}', ha='center', fontsize=11, fontweight='bold')
ax.axhline(bl_white, color='#4878CF', linestyle='--', linewidth=1.5,
           label=f'White baseline ({bl_white:.3f})')
ax.set_ylabel('AUPRC')
ax.set_title('Black patients — temperature completely removed')
ax.text(0.5, 0.05, f'delta={sb_black-bl_black:+.4f}  p={pb:.3f}',
        ha='center', transform=ax.transAxes, fontsize=10,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))
ax.set_ylim(0, max(vals + [bl_white])*1.15)
ax.legend(fontsize=9)

# PR curves
ax = axes[1]
yb = y[mask_black]
prec_base, rec_base, _ = precision_recall_curve(yb, oof_baseline[mask_black])
prec_scen, rec_scen, _ = precision_recall_curve(yb, oof_scen_b[mask_black])
ax.plot(rec_base, prec_base, color='#D65F5F', linewidth=2,
        label=f'Black baseline (AUPRC={bl_black:.3f})')
ax.plot(rec_scen, prec_scen, color='#8e44ad', linewidth=2, linestyle='--',
        label=f'Temperature removed (AUPRC={sb_black:.3f})')
ax.axhline(yb.mean(), color='gray', linestyle=':', alpha=0.6,
           label=f'Prevalence ({yb.mean():.2f})')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('PR curves — Black patients')
ax.legend(fontsize=9); ax.set_xlim(0,1); ax.set_ylim(0,1)

plt.suptitle('Scenario B: Complete temperature removal for Black patients', fontsize=13)
plt.tight_layout()
plt.savefig(SIM_DIR / 'fig_scenario_b_temperature.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_scenario_b_temperature.png')

In [ ]:
# Figure 4: Scenario C — Heart rate noise 
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

noise_levels = HR_NOISE_LEVELS
hr_black     = [hr_results[s]['black'] for s in noise_levels]
hr_white     = [hr_results[s]['white'] for s in noise_levels]
hr_p         = [hr_results[s].get('p', 1.0) for s in noise_levels]

ax = axes[0]
ax.plot(noise_levels, hr_black, 'o-', color='#D65F5F', linewidth=2.5,
        markersize=8, label='Black patients (noise added)')
ax.axhline(bl_white, color='#4878CF', linestyle='--', linewidth=1.5,
           label=f'White baseline ({bl_white:.3f})')
ax.axhline(bl_black, color='#D65F5F', linestyle=':', linewidth=1.5,
           label=f'Black baseline ({bl_black:.3f})')
for s, ap, p in zip(noise_levels, hr_black, hr_p):
    if p < 0.05:
        ax.plot(s, ap, '*', color='black', markersize=14, zorder=5)
ax.set_xlabel('Heart rate noise (Gaussian std, bpm)')
ax.set_ylabel('AUPRC')
ax.set_title('Scenario C — Heart rate noise injection\n(* = significant vs baseline, p<0.05)')
ax.legend(fontsize=9)
ax.set_xticks(noise_levels)

ax = axes[1]
gaps = [bl_white - hr_results[s]['black'] for s in noise_levels]
ax.bar(noise_levels, gaps, color='#e67e22', alpha=0.75, edgecolor='white')
ax.axhline(bl_white - bl_black, color='gray', linestyle='--',
           linewidth=1.5, label=f'Baseline gap ({bl_white-bl_black:.3f})')
ax.set_xlabel('Heart rate noise (bpm std)')
ax.set_ylabel('White AUPRC − Black AUPRC')
ax.set_title('Growing disparity gap as noise increases')
ax.legend(fontsize=9)
ax.set_xticks(noise_levels)

plt.suptitle('Scenario C: Heart rate noise bias against Black patients', fontsize=13)
plt.tight_layout()
plt.savefig(SIM_DIR / 'fig_scenario_c_hr_noise.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_scenario_c_hr_noise.png')

In [ ]:
# Figure 5: All scenarios together — overview panel 
fig, ax = plt.subplots(figsize=(13, 6))

# Collect all results into one plot showing Black AUPRC under each condition
conditions = ['Baseline']
black_vals = [bl_black]
colors_bar = ['#7f8c8d']

# Imputation exp 1
conditions.append('Exp 1\n(Black median\nimputation)')
black_vals.append(e1_black)
colors_bar.append('#6ACC65')

# Scenario B (temperature)
conditions.append('Scenario B\n(Temperature\nremoved)')
black_vals.append(sb_black)
colors_bar.append('#8e44ad')

# SpO2 shifts
for s in [1, 3, 5]:
    conditions.append(f'Scenario A\n(SpO2 -{s}pt)')
    black_vals.append(spo2_results[s]['black'])
    colors_bar.append('#c0392b')

# HR noise
for s in [5, 10, 20]:
    conditions.append(f'Scenario C\n(HR ±{s} bpm)')
    black_vals.append(hr_results[s]['black'])
    colors_bar.append('#e67e22')

x = np.arange(len(conditions))
bars = ax.bar(x, black_vals, color=colors_bar, edgecolor='white', alpha=0.85)
ax.axhline(bl_black, color='#D65F5F', linestyle='--', linewidth=1.5,
           label=f'Black baseline ({bl_black:.3f})')
ax.axhline(bl_white, color='#4878CF', linestyle='--', linewidth=1.5,
           label=f'White baseline ({bl_white:.3f})')

for bar, val in zip(bars, black_vals):
    ax.text(bar.get_x()+bar.get_width()/2, val+0.003,
            f'{val:.3f}', ha='center', fontsize=8, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(conditions, fontsize=8)
ax.set_ylabel('AUPRC — Black patients')
ax.set_title('Black patient AUPRC across all experiments and bias scenarios\n'
             'LSTM model weights unchanged throughout', fontsize=12)
ax.legend(fontsize=10)

# Legend patches
import matplotlib.patches as mpatches
legend_patches = [
    mpatches.Patch(color='#7f8c8d', label='Baseline'),
    mpatches.Patch(color='#6ACC65', label='Imputation (Black median)'),
    mpatches.Patch(color='#8e44ad', label='Temperature removed'),
    mpatches.Patch(color='#c0392b', label='SpO2 shift'),
    mpatches.Patch(color='#e67e22', label='Heart rate noise'),
]
ax.legend(handles=legend_patches, fontsize=9, loc='lower left')

plt.tight_layout()
plt.savefig(SIM_DIR / 'fig_all_scenarios_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_all_scenarios_overview.png')

---
## 11. Summary table

In [ ]:
summary_rows = [
    {'experiment': 'Baseline',
     'black_auprc': bl_black, 'white_auprc': bl_white,
     'delta_black': 0.0, 'gap': bl_white-bl_black, 'p_value': None},
    {'experiment': 'Exp 1 — Black median imputation',
     'black_auprc': e1_black, 'white_auprc': bl_white,
     'delta_black': e1_black-bl_black, 'gap': bl_white-e1_black, 'p_value': p1},
    {'experiment': 'Exp 2 — White degraded to Black rate',
     'black_auprc': bl_black, 'white_auprc': e2_white,
     'delta_black': 0.0, 'gap': e2_white-bl_black, 'p_value': p2},
    {'experiment': 'Scenario B — Temperature removed (Black)',
     'black_auprc': sb_black, 'white_auprc': bl_white,
     'delta_black': sb_black-bl_black, 'gap': bl_white-sb_black, 'p_value': pb},
]

for s in SPO2_SHIFTS[1:]:
    summary_rows.append({
        'experiment': f'Scenario A — SpO2 shift -{s}pt (Black)',
        'black_auprc': spo2_results[s]['black'],
        'white_auprc': spo2_results[s]['white'],
        'delta_black': spo2_results[s]['black'] - bl_black,
        'gap': spo2_results[s]['white'] - spo2_results[s]['black'],
        'p_value': spo2_results[s]['p'],
    })

for s in HR_NOISE_LEVELS[1:]:
    summary_rows.append({
        'experiment': f'Scenario C — HR noise std={s}bpm (Black)',
        'black_auprc': hr_results[s]['black'],
        'white_auprc': hr_results[s]['white'],
        'delta_black': hr_results[s]['black'] - bl_black,
        'gap': hr_results[s]['white'] - hr_results[s]['black'],
        'p_value': hr_results[s]['p'],
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(SIM_DIR / 'simulation_summary.csv', index=False)

print('=' * 75)
print('SIMULATION SUMMARY')
print('=' * 75)
print(summary_df[['experiment','black_auprc','white_auprc',
                  'delta_black','gap','p_value']].to_string(index=False))
print(f'\nAll outputs saved to: {SIM_DIR}')
for f in sorted(SIM_DIR.glob('*')):
    print(f'  {f.name}')